# Skill Co-occurrence Network

Builds a skill co-occurrence network from `mia_skills_long.csv` (long-format
url/skill pairs), compares the AI-mention vs non-AI-mention networks, and
saves the edge lists + comparison plot.

This is the third independent text-analysis method in the project (after
NMF topic modeling and TF-IDF term comparison) — see Inferences.md,
"Text analysis — skill co-occurrence network" for the interpretation.

**Inputs:** `data/mia_skills_long.csv`, `data/mia_postings_final2_fixed2.csv`
**Outputs:** `data/mia_skill_network_comparison.png`,
`data/mia_skill_cooccurrence_edges.csv`, `data/mia_skill_group_exclusive.csv`

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
SKILLS_CSV = "data/mia_skills_long.csv"
MIN_DOC_FREQ = 5    # a skill must appear in at least this many postings to keep
MIN_COOCCUR = 10    # starting threshold — raised below once the full graph is built

skills_long = pd.read_csv(SKILLS_CSV)
print(f"{len(skills_long):,} rows, {skills_long['url'].nunique():,} postings, {skills_long['skill'].nunique()} skills")

In [ ]:
# Pivot to a wide binary matrix: rows = postings, columns = skills, 1 = mentioned
skill_wide = pd.crosstab(skills_long["url"], skills_long["skill"])
skill_wide = (skill_wide > 0).astype(int)

In [ ]:
# Drop rare skills below MIN_DOC_FREQ
skill_freq = skill_wide.sum(axis=0)
keep_skills = skill_freq[skill_freq >= MIN_DOC_FREQ].index
skill_wide = skill_wide[keep_skills]
print(f"Matrix: {skill_wide.shape[0]:,} postings x {skill_wide.shape[1]} skills (after MIN_DOC_FREQ filter)")

In [ ]:
# Co-occurrence = how many postings mention BOTH skill A and skill B.
# .to_numpy().copy() before fill_diagonal — pandas' .values can return a
# read-only view, which fill_diagonal can't write into in place.
cooccur = skill_wide.T.dot(skill_wide).astype(int)
arr = cooccur.to_numpy().copy()
np.fill_diagonal(arr, 0)
cooccur = pd.DataFrame(arr, index=cooccur.index, columns=cooccur.columns)

top_pair = cooccur.stack().sort_values(ascending=False).index[0]
top_val = cooccur.stack().sort_values(ascending=False).iloc[0]
print(f"Top co-occurring pair: {top_pair} — {top_val} postings")

In [ ]:
import networkx as nx   # if not installed: pip install networkx

A first pass at `MIN_COOCCUR = 10` produced 357 edges on 42 nodes — 41% of
all possible pairs, an unreadable hairball when plotted. Threshold raised to
40, which brings it down to a workable 153 edges.

In [ ]:
MIN_COOCCUR = 40

G = nx.Graph()
for skill in skill_wide.columns:
    G.add_node(skill, freq=int(skill_freq[skill]))

skills = cooccur.columns
for i, a in enumerate(skills):
    for b in skills[i+1:]:
        weight = cooccur.loc[a, b]
        if weight >= MIN_COOCCUR:
            G.add_edge(a, b, weight=int(weight))

print(f"Graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges (co-occurrence >= {MIN_COOCCUR})")

In [ ]:
import matplotlib.pyplot as plt

# Drop isolated nodes (skills that cleared MIN_DOC_FREQ but lost all edges at this threshold)
G_plot = G.copy()
G_plot.remove_nodes_from(list(nx.isolates(G_plot)))
print(f"Plotting {G_plot.number_of_nodes()} connected nodes")

pos = nx.spring_layout(G_plot, k=0.5, seed=42, weight="weight")

node_sizes = [G_plot.nodes[n]["freq"] * 2 for n in G_plot.nodes]
edge_weights = [G_plot[u][v]["weight"] / 20 for u, v in G_plot.edges]

plt.figure(figsize=(14, 10))
nx.draw_networkx_nodes(G_plot, pos, node_size=node_sizes, node_color="#4C72B0", alpha=0.85)
nx.draw_networkx_edges(G_plot, pos, width=edge_weights, alpha=0.3, edge_color="gray")
nx.draw_networkx_labels(G_plot, pos, font_size=9)
title_text = "Skill Co-occurrence Network (edges >= " + str(MIN_COOCCUR) + " co-occurrences)"
plt.title(title_text)
plt.axis("off")
plt.tight_layout()
plt.show()

## Split by AI-mention vs non-AI-mention

Joins the skill long-table to the main postings file on `url` to get the
`mentions_ai` flag, then builds two separate networks for comparison.

In [ ]:
POSTINGS_CSV = "data/mia_postings_final2_fixed2.csv"

postings = pd.read_csv(POSTINGS_CSV, usecols=["url", "mentions_ai"])
skills_flagged = skills_long.merge(postings, on="url", how="left")
print(f"{skills_flagged['mentions_ai'].isna().sum()} skill-rows failed to match a posting (should be 0 or small)")

skills_ai = skills_flagged[skills_flagged["mentions_ai"] == True]
skills_non_ai = skills_flagged[skills_flagged["mentions_ai"] == False]
print(f"AI-mention: {skills_ai['url'].nunique()} postings, {len(skills_ai)} skill mentions")
print(f"Non-AI-mention: {skills_non_ai['url'].nunique()} postings, {len(skills_non_ai)} skill mentions")

In [ ]:
def build_skill_graph(skills_df, min_doc_freq=5, min_cooccur=15):
    wide = pd.crosstab(skills_df["url"], skills_df["skill"])
    wide = (wide > 0).astype(int)
    freq = wide.sum(axis=0)
    keep = freq[freq >= min_doc_freq].index
    wide = wide[keep]

    co = wide.T.dot(wide).astype(int)
    arr = co.to_numpy().copy()
    np.fill_diagonal(arr, 0)
    co = pd.DataFrame(arr, index=co.index, columns=co.columns)

    g = nx.Graph()
    for s in wide.columns:
        g.add_node(s, freq=int(freq[s]))
    cols = co.columns
    for i, a in enumerate(cols):
        for b in cols[i+1:]:
            w = co.loc[a, b]
            if w >= min_cooccur:
                g.add_edge(a, b, weight=int(w))
    return g

AI-mention group is denser than non-AI (more skills per posting on average —
7,302 mentions across 1,763 postings vs 5,764 across 2,019), so it needs a
higher threshold to stay readable: 30 for AI-mention, 15 for non-AI.

In [ ]:
G_ai = build_skill_graph(skills_ai, min_doc_freq=5, min_cooccur=30)
G_non_ai = build_skill_graph(skills_non_ai, min_doc_freq=5, min_cooccur=15)

print(f"AI-mention graph: {G_ai.number_of_nodes()} nodes, {G_ai.number_of_edges()} edges")
print(f"Non-AI graph: {G_non_ai.number_of_nodes()} nodes, {G_non_ai.number_of_edges()} edges")

# Skills exclusive to one group's network at these thresholds.
# All 11 group-exclusive skills are AI-mention-only — zero are non-AI-only.
ai_only = set(G_ai.nodes) - set(G_non_ai.nodes)
non_ai_only = set(G_non_ai.nodes) - set(G_ai.nodes)
print(f"AI-only skills: {sorted(ai_only)}")
print(f"Non-AI-only skills: {sorted(non_ai_only)}")

In [ ]:
def plot_graph(g, title, ax):
    g2 = g.copy()
    g2.remove_nodes_from(list(nx.isolates(g2)))
    pos = nx.spring_layout(g2, k=0.5, seed=42, weight="weight")
    sizes = [g2.nodes[n]["freq"] * 2 for n in g2.nodes]
    widths = [g2[u][v]["weight"] / 15 for u, v in g2.edges]
    nx.draw_networkx_nodes(g2, pos, node_size=sizes, node_color="#4C72B0", alpha=0.85, ax=ax)
    nx.draw_networkx_edges(g2, pos, width=widths, alpha=0.3, edge_color="gray", ax=ax)
    nx.draw_networkx_labels(g2, pos, font_size=7, ax=ax)
    ax.set_title(title)
    ax.axis("off")

fig, axes = plt.subplots(1, 2, figsize=(20, 10))
plot_graph(G_ai, "AI-mention skill network", axes[0])
plot_graph(G_non_ai, "Non-AI-mention skill network", axes[1])
plt.tight_layout()
plt.savefig("data/mia_skill_network_comparison.png", dpi=200, bbox_inches="tight")
plt.show()
print("Saved to data/mia_skill_network_comparison.png")

In [ ]:
def edges_to_df(g, label):
    rows = [{"group": label, "skill_a": u, "skill_b": v, "cooccurrence": d["weight"]}
            for u, v, d in g.edges(data=True)]
    return pd.DataFrame(rows).sort_values("cooccurrence", ascending=False)

edges_ai = edges_to_df(G_ai, "AI-mention")
edges_non_ai = edges_to_df(G_non_ai, "Non-AI-mention")
all_edges = pd.concat([edges_ai, edges_non_ai], ignore_index=True)
all_edges.to_csv("data/mia_skill_cooccurrence_edges.csv", index=False)
print(f"Saved {len(all_edges)} edges to mia_skill_cooccurrence_edges.csv")

In [ ]:
ai_only_df = pd.DataFrame({"skill": sorted(ai_only), "group": "AI-mention only"})
ai_only_df.to_csv("data/mia_skill_group_exclusive.csv", index=False)
print(f"Saved {len(ai_only_df)} AI-exclusive skills to mia_skill_group_exclusive.csv")